# RTU Senior Tutor — QLoRA Fine-Tuning on Google Colab (Free T4 GPU)

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** on **748 authentic RTU past examination examples** across Semesters 3 to 8.

### End-to-End Workflow:
1. **Environment Setup**: Verify free T4 GPU & install Hugging Face / TRL dependencies.
2. **Upload Dataset**: Upload `rtu_train_dataset_expanded.jsonl` (~1 MB).
3. **Setup Training Script**: Self-contained `train_qlora_colab.py` generated automatically.
4. **QLoRA Fine-Tuning**: 4-bit NF4 quantized training (~15 minutes on T4).
5. **Merge Weights**: Merge LoRA adapter into 16-bit standalone weights.
6. **GGUF Export**: Convert with `llama.cpp` to `rtu_senior_tutor_q4.gguf`.
7. **Deploy to Local Ollama**: Run 100% offline on your laptop as `rtu-senior`.

--- 
## Step 1: Verify Free T4 GPU Runtime
Make sure your Colab runtime is set to **T4 GPU** (*Runtime > Change runtime type > T4 GPU*).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name   : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: GPU not detected. Please go to Runtime > Change runtime type and select T4 GPU!")

--- 
## Step 2: Install Required Libraries

In [ ]:
!pip install -q torch transformers datasets trl peft bitsandbytes accelerate

--- 
## Step 3: Upload `rtu_train_dataset_expanded.jsonl`
Run the cell below to upload `rtu_train_dataset_expanded.jsonl` (~1 MB) from your local computer.

In [ ]:
import os

dataset_name = "rtu_train_dataset_expanded.jsonl"
if not os.path.exists(dataset_name):
    print(f"Uploading {dataset_name}...")
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as e:
        print(f"Upload error: {e}. You can also drag-and-drop the file into the Files panel on the left.")

if os.path.exists(dataset_name):
    size_mb = os.path.getsize(dataset_name) / (1024 * 1024)
    print(f"SUCCESS: {dataset_name} is ready ({size_mb:.2f} MB).")
else:
    print(f"WARNING: {dataset_name} is not present yet. Please upload it before proceeding.")

--- 
## Step 4: Write `train_qlora_colab.py` Training Script
The cell below automatically creates `train_qlora_colab.py` directly inside your Colab environment.

In [ ]:
# Generate train_qlora_colab.py directly inside the Colab session
script_code = '"""\n==============================================================================\nRTU Senior Tutor — QLoRA Fine-Tuning Script for Google Colab (Free T4 GPU)\n==============================================================================\n\nPROJECT CONTEXT\n---------------\nThis script fine-tunes Qwen2.5-1.5B-Instruct using QLoRA on 200 RTU-specific\ntraining examples (seed_dataset_200.jsonl) to create the "Senior Tutor" persona\nfor the RTU AI Academic Assistant.\n\nThe fine-tuned model will be used as the offline/local LLM provider in the\nbackend via Ollama (see: backend/app/llm/ollama_provider.py).\n\nGOOGLE COLAB USAGE\n------------------\nStep 1 — Upload both files to Colab:\n    - train_qlora_colab.py\n    - seed_dataset_200.jsonl\n\nStep 2 — Install dependencies (run in a Colab cell):\n    !pip install -q torch transformers datasets trl peft bitsandbytes accelerate\n\nStep 3 — Run training (run in a Colab cell):\n    !python train_qlora_colab.py\n\nStep 4 — After training, merge + export (run in a Colab cell):\n    !python train_qlora_colab.py --merge_only\n\nMODEL SPECS (Free T4 GPU — 15 GB VRAM)\n----------------------------------------\n  Base model  : Qwen/Qwen2.5-1.5B-Instruct (1.5B params, ~3.0 GB in 4-bit)\n  LoRA rank   : r=16, alpha=32 (18.4M trainable params, 1.18% of total)\n  Batch       : 4 x 4 grad_accum = 16 effective\n  Epochs      : 3\n  VRAM usage  : ~5–6 GB (safe for free T4)\n  Time        : ~12–15 minutes on T4\n\nWHAT THIS PRODUCES\n------------------\n  ./rtu_lora_adapter/     — LoRA adapter weights\n  ./rtu_merged_model/     — Full merged model (for GGUF export)\n  ./rtu_senior_tutor_q4.gguf — Final GGUF for Ollama (after llama.cpp step)\n\nOLLAMA USAGE (after GGUF export, on your local machine)\n--------------------------------------------------------\n    ollama create rtu-senior -f Modelfile\n    # In backend: set LLM_PROVIDER=ollama, OLLAMA_MODEL=rtu-senior\n==============================================================================\n"""\n\nimport os\nimport sys\nimport json\nimport inspect\nimport argparse\nimport torch\nfrom pathlib import Path\nfrom datasets import load_dataset\nfrom transformers import (\n    AutoModelForCausalLM,\n    AutoTokenizer,\n    BitsAndBytesConfig,\n)\nfrom peft import LoraConfig, prepare_model_for_kbit_training\n\n\n# ==============================================================================\n# CONFIGURATION — Change these if needed\n# ==============================================================================\n\nDEFAULT_BASE_MODEL  = "Qwen/Qwen2.5-1.5B-Instruct"\nDEFAULT_DATASET     = "rtu_train_dataset_expanded.jsonl" if Path("rtu_train_dataset_expanded.jsonl").exists() or (Path(__file__).parent / "rtu_train_dataset_expanded.jsonl").exists() else "seed_dataset_200.jsonl"\nDEFAULT_LORA_DIR    = "./rtu_lora_adapter"\nDEFAULT_MERGED_DIR  = "./rtu_merged_model"\nDEFAULT_EPOCHS      = 3\nDEFAULT_BATCH_SIZE  = 4\nDEFAULT_GRAD_ACCUM  = 4\nDEFAULT_LR          = 2e-4\nDEFAULT_MAX_SEQ_LEN = 2048\n\n\n# ==============================================================================\n# ARGUMENT PARSING\n# ==============================================================================\n\ndef parse_args() -> argparse.Namespace:\n    p = argparse.ArgumentParser(description="RTU Senior Tutor QLoRA Fine-Tuning")\n    p.add_argument("--base_model",   default=DEFAULT_BASE_MODEL)\n    p.add_argument("--dataset_path", default=DEFAULT_DATASET)\n    p.add_argument("--output_dir",   default=DEFAULT_LORA_DIR)\n    p.add_argument("--merged_dir",   default=DEFAULT_MERGED_DIR)\n    p.add_argument("--epochs",       type=int,   default=DEFAULT_EPOCHS)\n    p.add_argument("--batch_size",   type=int,   default=DEFAULT_BATCH_SIZE)\n    p.add_argument("--grad_accum",   type=int,   default=DEFAULT_GRAD_ACCUM)\n    p.add_argument("--lr",           type=float, default=DEFAULT_LR)\n    p.add_argument("--max_seq_len",  type=int,   default=DEFAULT_MAX_SEQ_LEN)\n    p.add_argument(\n        "--merge_only",\n        action="store_true",\n        help="Skip training; only merge existing LoRA adapter and export."\n    )\n    return p.parse_args()\n\n\n# ==============================================================================\n# HELPERS\n# ==============================================================================\n\ndef banner(text: str):\n    print("\\n" + "=" * 64)\n    print(f"  {text}")\n    print("=" * 64)\n\n\ndef device_info() -> str:\n    if torch.cuda.is_available():\n        name = torch.cuda.get_device_name(0)\n        vram = torch.cuda.get_device_properties(0).total_memory / 1e9\n        return f"CUDA — {name} ({vram:.1f} GB VRAM)"\n    return "CPU (Warning: training will be very slow!)"\n\n\ndef try_init(cls, fixed_kwargs: dict, variant_list: list):\n    """\n    Attempt to construct `cls(**fixed_kwargs, **variant)` for each dict in\n    variant_list, returning the first successful instance.\n\n    This handles TRL API changes across versions without crashing:\n      - SFTConfig: max_length vs max_seq_length vs dataset_text_field\n      - SFTTrainer: processing_class vs tokenizer\n    """\n    last_error = None\n    for variant in variant_list:\n        try:\n            return cls(**fixed_kwargs, **variant)\n        except TypeError as e:\n            last_error = e\n            continue\n    raise RuntimeError(\n        f"Could not construct {cls.__name__} with any parameter variant.\\n"\n        f"Last error: {last_error}\\n"\n        f"TRL version may be unsupported. Try: pip install -q trl==0.11.4"\n    )\n\n\n# ==============================================================================\n# STEP 1 — LOAD TOKENIZER\n# ==============================================================================\n\ndef load_tokenizer(model_name: str) -> AutoTokenizer:\n    print("\\n[1/5] Loading tokenizer ...")\n    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "right"   # Required for SFT causal left-padding fix\n    print(f"      Vocab size    : {tok.vocab_size:,}")\n    print(f"      Pad token     : \'{tok.pad_token}\'")\n    return tok\n\n\n# ==============================================================================\n# STEP 2 — LOAD QUANTIZED BASE MODEL\n# ==============================================================================\n\ndef load_model(model_name: str) -> AutoModelForCausalLM:\n    print("\\n[2/5] Loading 4-bit NF4 quantized base model ...")\n    bnb_config = BitsAndBytesConfig(\n        load_in_4bit=True,\n        bnb_4bit_quant_type="nf4",\n        bnb_4bit_compute_dtype=torch.float16,\n        bnb_4bit_use_double_quant=True,\n    )\n    model = AutoModelForCausalLM.from_pretrained(\n        model_name,\n        quantization_config=bnb_config,\n        device_map="auto",\n        trust_remote_code=True,\n        # Force ALL non-quantized layers (norms, embeddings) to float16.\n        # Qwen2.5 defaults to bfloat16 which crashes on T4 GPU with fp16 AMP.\n        torch_dtype=torch.float16,\n    )\n    model = prepare_model_for_kbit_training(model)\n\n    # Hard-cast any remaining bfloat16 tensors → float16.\n    # paged_adamw_8bit can leave some optimizer-state tensors in bfloat16\n    # which makes the CUDA AMP grad scaler raise NotImplementedError on T4.\n    bf16_count = 0\n    for param in model.parameters():\n        if param.dtype == torch.bfloat16:\n            param.data = param.data.to(torch.float16)\n            bf16_count += 1\n    if bf16_count:\n        print(f"      Recast {bf16_count} bfloat16 tensors → float16 for T4 compatibility.")\n    total_params = sum(p.numel() for p in model.parameters())\n    print(f"      Total params  : {total_params / 1e6:.1f}M")\n    return model\n\n\n# ==============================================================================\n# STEP 3 — BUILD LORA CONFIG (not applied manually — SFTTrainer does it)\n# ==============================================================================\n\ndef build_lora_config() -> LoraConfig:\n    print("\\n[3/5] Building LoRA config (r=16, alpha=32) ...")\n    cfg = LoraConfig(\n        r=16,\n        lora_alpha=32,\n        lora_dropout=0.05,\n        bias="none",\n        task_type="CAUSAL_LM",\n        # Target all projection layers in Qwen2.5 / LLaMA-style architectures\n        target_modules=[\n            "q_proj", "k_proj", "v_proj", "o_proj",\n            "gate_proj", "up_proj", "down_proj",\n        ],\n    )\n    print("      LoRA ready. SFTTrainer will apply it automatically.")\n    return cfg\n\n\n# ==============================================================================\n# STEP 4 — LOAD & FORMAT DATASET\n# ==============================================================================\n\ndef load_and_format_dataset(dataset_path: str, tokenizer: AutoTokenizer):\n    """\n    Loads seed_dataset_200.jsonl and applies the tokenizer\'s chat template to\n    each example, producing a single \'text\' column that SFTTrainer trains on.\n\n    Dataset format (each line):\n        {\n          "task": "explain_topic",\n          "messages": [\n            {"role": "system",    "content": "..."},\n            {"role": "user",      "content": "..."},\n            {"role": "assistant", "content": "..."}\n          ]\n        }\n    """\n    print(f"\\n[4/5] Loading dataset from: {dataset_path}")\n\n    actual_path = Path(dataset_path)\n    if not actual_path.exists():\n        script_relative = Path(__file__).parent / dataset_path\n        if script_relative.exists():\n            actual_path = script_relative\n        else:\n            fallback = Path(__file__).parent / "rtu_train_dataset_expanded.jsonl"\n            if fallback.exists():\n                actual_path = fallback\n            else:\n                fallback_seed = Path(__file__).parent / "seed_dataset_200.jsonl"\n                if fallback_seed.exists():\n                    actual_path = fallback_seed\n                else:\n                    raise FileNotFoundError(\n                        f"Dataset not found at: {dataset_path} or {script_relative}\\n"\n                        "Please ensure rtu_train_dataset_expanded.jsonl is present."\n                    )\n\n    raw = load_dataset("json", data_files=str(actual_path), split="train")\n    print(f"      Raw examples  : {len(raw)}")\n\n    def apply_chat_template(example):\n        text = tokenizer.apply_chat_template(\n            example["messages"],\n            tokenize=False,\n            add_generation_prompt=False,\n        )\n        return {"text": text}\n\n    formatted = raw.map(apply_chat_template, remove_columns=raw.column_names)\n    print(f"      Formatted     : {len(formatted)} conversation turns")\n\n    # Print a sample so we can verify the format looks right\n    sample = formatted[0]["text"]\n    print(f"      Sample (first 200 chars): {repr(sample[:200])}")\n\n    return formatted\n\n\n# ==============================================================================\n# STEP 5 — CONFIGURE SFTCONFIG (version-agnostic)\n# ==============================================================================\n\ndef build_sft_config(args: argparse.Namespace):\n    """\n    Build SFTConfig in a way that works across TRL 0.8–0.14+.\n\n    Key API changes across TRL versions:\n      - warmup_ratio removed → use warmup_steps\n      - dataset_text_field moved from SFTTrainer to SFTConfig, then removed\n      - max_seq_length renamed to max_length in SFTConfig\n      - TrainingArguments subclassed by SFTConfig in newer versions\n    """\n    from trl import SFTConfig\n\n    # Base kwargs that are stable across all recent TRL versions\n    base = dict(\n        output_dir=args.output_dir,\n        per_device_train_batch_size=args.batch_size,\n        gradient_accumulation_steps=args.grad_accum,\n        learning_rate=args.lr,\n        lr_scheduler_type="cosine",\n        warmup_steps=10,\n        num_train_epochs=args.epochs,\n        logging_steps=5,\n        save_strategy="epoch",\n        # fp16=False, bf16=False → train in float32 AMP-free.\n        # T4 GPU\'s CUDA AMP scaler does not support bfloat16, and Qwen2.5\n        # has residual bfloat16 tensors that crash the grad scaler.\n        # float32 training on 200 examples is ~18 min on T4 — perfectly fine.\n        fp16=False,\n        bf16=False,\n        optim="adamw_torch",    # paged_adamw_8bit creates bfloat16 opt states\n        report_to="none",\n        dataloader_num_workers=0,\n    )\n\n    # Variants to try — ordered from newest TRL to oldest\n    # max_length (new)        → TRL >= 0.12\n    # max_seq_length (old)    → TRL 0.8–0.11\n    # dataset_text_field      → needed only in some middle versions\n    variants = [\n        {"max_length": args.max_seq_len},\n        {"max_seq_length": args.max_seq_len},\n        {"max_length": args.max_seq_len, "dataset_text_field": "text"},\n        {"max_seq_length": args.max_seq_len, "dataset_text_field": "text"},\n        {},   # Absolute fallback — no extra params at all\n    ]\n\n    cfg = try_init(SFTConfig, base, variants)\n    print(f"\\n[5a] SFTConfig built successfully.")\n    return cfg\n\n\n# ==============================================================================\n# STEP 5b — BUILD SFTTRAINER (version-agnostic)\n# ==============================================================================\n\ndef build_trainer(model, dataset, sft_config, lora_config, tokenizer):\n    """\n    Build SFTTrainer in a way that works across TRL versions.\n\n    Key API changes:\n      - processing_class (new, >= TRL 0.12)  replaces  tokenizer (old)\n      - peft_config passed here wraps the model — do NOT call get_peft_model()\n        manually beforehand, or you will get:\n        ValueError: You passed a PeftModel instance together with a peft_config\n    """\n    from trl import SFTTrainer\n\n    base = dict(\n        model=model,\n        train_dataset=dataset,\n        args=sft_config,\n        peft_config=lora_config,   # SFTTrainer wraps model internally\n    )\n\n    # processing_class (new) → tokenizer (old) → no tokenizer kwarg\n    variants = [\n        {"processing_class": tokenizer},\n        {"tokenizer": tokenizer},\n        {},\n    ]\n\n    trainer = try_init(SFTTrainer, base, variants)\n    print(f"[5b] SFTTrainer built successfully.")\n\n    # Print trainable parameter count after PEFT wrapping\n    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)\n    total     = sum(p.numel() for p in trainer.model.parameters())\n    print(f"     Trainable params : {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")\n    return trainer\n\n\n# ==============================================================================\n# MERGE LORA & EXPORT (run after training)\n# ==============================================================================\n\ndef merge_and_save(args: argparse.Namespace, tokenizer: AutoTokenizer):\n    """\n    Merge LoRA adapter weights into the base model and save a full merged model.\n    Uses PeftModel directly (not AutoPeftModelForCausalLM) to avoid the\n    torchao version-check bug in PEFT >= 0.14 on Colab.\n    """\n    from peft import PeftModel\n\n    banner("Merging LoRA adapter into base model ...")\n    print(f"  Adapter dir  : {args.output_dir}")\n    print(f"  Merged dir   : {args.merged_dir}")\n    print("  Loading fresh base model on CPU (avoids VRAM overflow) ...")\n\n    # Load the raw base model in float16 on CPU\n    base_model = AutoModelForCausalLM.from_pretrained(\n        args.base_model,\n        torch_dtype=torch.float16,\n        device_map="cpu",\n        trust_remote_code=True,\n    )\n\n    # Load and attach LoRA adapter\n    print("  Attaching LoRA adapter ...")\n    try:\n        peft_model = PeftModel.from_pretrained(\n            base_model,\n            args.output_dir,\n            torch_dtype=torch.float16,\n        )\n    except ImportError as e:\n        if "torchao" in str(e):\n            print("  Detected torchao version incompatibility. Falling back to AutoPeftModelForCausalLM...")\n            from peft import AutoPeftModelForCausalLM\n            peft_model = AutoPeftModelForCausalLM.from_pretrained(\n                args.output_dir,\n                device_map="cpu",\n                torch_dtype=torch.float16,\n            )\n        else:\n            raise e\n\n    # Merge adapter weights into base model and unload adapter scaffolding\n    print("  Merging weights ...")\n    merged_model = peft_model.merge_and_unload()\n    merged_model.save_pretrained(args.merged_dir)\n    tokenizer.save_pretrained(args.merged_dir)\n    print(f"  Merged model saved to: {args.merged_dir}")\n\n    print("""\nNext step — Convert to GGUF (run in Colab):\n  !git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp\n  !cd /content/llama.cpp && pip install -r requirements.txt\n  !python /content/llama.cpp/convert_hf_to_gguf.py ./rtu_merged_model \\\\\n      --outfile rtu_senior_tutor_fp16.gguf --outtype f16\n  !./llama.cpp/llama-quantize rtu_senior_tutor_fp16.gguf \\\\\n      rtu_senior_tutor_q4.gguf Q4_K_M\n\nThen download rtu_senior_tutor_q4.gguf to your laptop and run:\n  ollama create rtu-senior -f Modelfile\n""")\n\n\n# ==============================================================================\n# MAIN\n# ==============================================================================\n\ndef main():\n    args = parse_args()\n\n    banner("RTU Senior Tutor — QLoRA Fine-Tuning")\n    print(f"  Base model  : {args.base_model}")\n    print(f"  Dataset     : {args.dataset_path}")\n    print(f"  Output dir  : {args.output_dir}")\n    print(f"  Device      : {device_info()}")\n    print(f"  Epochs      : {args.epochs}")\n    print(f"  Batch size  : {args.batch_size} (x{args.grad_accum} grad_accum = {args.batch_size * args.grad_accum} effective)")\n    print(f"  Max seq len : {args.max_seq_len}")\n\n    if args.merge_only:\n        # --- Merge-only mode: load tokenizer and merge existing adapter ---\n        tokenizer = load_tokenizer(args.base_model)\n        merge_and_save(args, tokenizer)\n        return\n\n    # --- Full training pipeline ---\n\n    # 1. Tokenizer\n    tokenizer = load_tokenizer(args.base_model)\n\n    # 2. Base model (4-bit quantized)\n    model = load_model(args.base_model)\n\n    # 3. LoRA config (applied by SFTTrainer internally)\n    lora_config = build_lora_config()\n\n    # 4. Dataset\n    dataset = load_and_format_dataset(args.dataset_path, tokenizer)\n\n    # 5. SFTConfig\n    banner("Step 5 — Configuring SFTTrainer & Starting Fine-Tuning")\n    sft_config = build_sft_config(args)\n\n    # 5b. SFTTrainer\n    trainer = build_trainer(model, dataset, sft_config, lora_config, tokenizer)\n\n    # 6. Train\n    banner("Training started — watch for loss values every 5 steps")\n    trainer.train()\n\n    # 7. Save LoRA adapter\n    banner("Saving LoRA adapter ...")\n    trainer.model.save_pretrained(args.output_dir)\n    tokenizer.save_pretrained(args.output_dir)\n    print(f"  Adapter saved to: {args.output_dir}")\n\n    # 8. Auto-merge after training\n    merge_and_save(args, tokenizer)\n\n    banner("SUCCESS — RTU Senior Tutor fine-tuning complete!")\n    print(f"""\nSummary:\n  LoRA adapter  -> {args.output_dir}/\n  Merged model  -> {args.merged_dir}/\n\nWhat to do next:\n  1. Convert rtu_merged_model/ to GGUF (see instructions above)\n  2. Download rtu_senior_tutor_q4.gguf to your laptop\n  3. Run: ollama create rtu-senior -f Modelfile\n  4. In backend .env: set LLM_PROVIDER=ollama and OLLAMA_MODEL=rtu-senior\n  5. The Senior Tutor persona is now your local offline AI!\n""")\n\n\nif __name__ == "__main__":\n    main()\n'

with open('train_qlora_colab.py', 'w', encoding='utf-8') as f:
    f.write(script_code)

print("SUCCESS: train_qlora_colab.py generated and verified!")

--- 
## Step 5: Run QLoRA Fine-Tuning
- **Base Model**: `Qwen/Qwen2.5-1.5B-Instruct`
- **Quantization**: 4-bit NormalFloat (NF4) with double quant
- **LoRA Parameters**: $r=16, \alpha=32$, targeting all linear projections
- **Duration**: ~15 minutes for 3 epochs on free Colab T4 GPU

In [ ]:
!python train_qlora_colab.py \
  --base_model "Qwen/Qwen2.5-1.5B-Instruct" \
  --dataset_path "rtu_train_dataset_expanded.jsonl" \
  --output_dir "./rtu_lora_adapter" \
  --epochs 3 \
  --batch_size 4 \
  --grad_accum 4 \
  --lr 2e-4 \
  --max_seq_len 2048

--- 
## Step 6: Merge LoRA Weights into Standalone 16-Bit Model

In [ ]:
!python train_qlora_colab.py \
  --merge_only \
  --output_dir "./rtu_lora_adapter" \
  --merged_dir "./rtu_merged_model"

--- 
## Step 7: Convert Merged Model to GGUF Format via `llama.cpp`
This converts the fine-tuned PyTorch model into high-efficiency 4-bit GGUF (`rtu_senior_tutor_q4.gguf`) for local Ollama inference.

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py ./rtu_merged_model \
  --outfile ./rtu_senior_tutor_q4.gguf \
  --outtype q4_k_m

--- 
## Step 8: Download GGUF to Your Laptop
Run this cell to automatically trigger the browser download for `rtu_senior_tutor_q4.gguf` (~1.1 GB).

In [ ]:
from google.colab import files
import os

if os.path.exists("rtu_senior_tutor_q4.gguf"):
    print("Downloading rtu_senior_tutor_q4.gguf to your computer...")
    files.download("rtu_senior_tutor_q4.gguf")
else:
    print("GGUF file not found. Ensure Step 7 completed successfully.")

--- 
## Step 9: Deploying to Local Ollama

After the download completes, place `rtu_senior_tutor_q4.gguf` on your computer and run:

### 1. Create `Modelfile`:
```dockerfile
FROM ./rtu_senior_tutor_q4.gguf
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
SYSTEM """You are Senior Tutor - an empathetic, high-scoring senior student from Rajasthan Technical University (RTU). You explain concepts in simple English or Hinglish, provide exam-oriented 10-mark structures (Definition, Diagram, Steps, Example, RTU Tip), grade student answers honestly with constructive feedback, and strictly refuse out-of-syllabus topics."""
```

### 2. Register with Ollama:
```bash
ollama create rtu-senior -f Modelfile
ollama run rtu-senior
```

### 3. Use in RTU AI Academic Assistant:
Set `OLLAMA_MODEL = "rtu-senior"` in `config.py` and run your assistant 100% locally with zero external API fees!